<a href="https://colab.research.google.com/github/thahsinj06/Fly-rank-ml-internship-work/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thahsinj06/Fly-rank-ml-internship-work/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1: Unit of analysis + time window

Think of each row as one CCTV snapshot of a webpage.

**One row = one pseudonymized content item for one pseudonymized client on one report date.**

The main table is `fact_content_daily_performance`, which stores daily page-performance observations.

For this notebook, I will investigate the **March 2026** partition (`month = '2026-03'`). I use a mid-panel month rather than the final month so that later performance can remain separate from the information available at the decision moment.

The eventual decision is:

> **Which content pages should receive higher priority for human refresh review?**

The goal is therefore to build a system that uses information available at the time of the decision, without looking into the future.

In [16]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("Hugging Face authentication ready.")

Token loaded: True
Hugging Face authentication ready.


###  2: Fields and evidence rules

I divide the fields into four groups.

**🔧 Features — evidence available for the model**

- `content_age_days` — tells us how old the content is.
- `days_since_last_update` — tells us how recently the content was changed.
- `impressions_90d` — describes recent search visibility.
- `avg_position` — describes recent search ranking.
- `ctr` — describes how often impressions resulted in clicks.

These features describe the page using information that could be available when making the refresh-priority decision.

**🎯 Label / proxy**

- `is_declining_label` — a proxy indicating observed decline.

This is not a guarantee that refreshing the page will improve it. It is an observed signal that can be used to evaluate whether the scoring approach identifies pages showing signs of decline.

**🧭 Context**

- `client_key` — identifies the pseudonymized client.
- `content_key` — identifies the content item.
- `report_date` — identifies when the observation occurred.
- `content_type` — provides contextual information about the page.

**🚫 Excluded**

I deliberately exclude future performance information and label-derived fields from the feature set.

These would not be available at the decision moment and could allow the model to see information derived from the outcome it is supposed to predict.

I also exclude `client_key` from predictive features because it identifies the client context rather than describing the performance characteristics of the content itself.

In [17]:
feature_fields = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
]

label_fields = [
    "is_declining_label"
]

context_fields = [
    "client_key",
    "content_key",
    "report_date",
    "content_type"
]

excluded_fields = [
    "future_performance",
    "label_derived_features",
    "client_key_as_predictor"
]

print("🔧 FEATURES")
print(feature_fields)

print("\n🎯 LABEL / PROXY")
print(label_fields)

print("\n🧭 CONTEXT")
print(context_fields)

print("\n🚫 EXCLUDED")
print(excluded_fields)

🔧 FEATURES
['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr']

🎯 LABEL / PROXY
['is_declining_label']

🧭 CONTEXT
['client_key', 'content_key', 'report_date', 'content_type']

🚫 EXCLUDED
['future_performance', 'label_derived_features', 'client_key_as_predictor']


## 3. Verify the data contract

I verify the contract with three checks on the March 2026 slice:

1. **Grain:** confirm that `report_date + client_hash_id + content_hash_id` uniquely identifies a row.
2. **Count and date window:** measure the number of rows and verify the first and last report dates.
3. **Availability:** count rows where Search Console data is explicitly available using `IS TRUE`.

These checks turn the assumptions in the data contract into measured observations.

### What the verification shows

The March 2026 slice contains **9,841,378 observations**, covering the full period from **2026-03-01 to 2026-03-31**.

The grain check found **0 duplicate combinations** of `report_date`, `client_hash_id`, and `content_hash_id`. This supports the documented grain that one row represents one content item for one client on one report date.

The availability check found **3,611,061 observations (36.69%)** where `gsc_data_available IS TRUE`. This means that a large portion of the March observations does not have GSC data marked as available.

I therefore treat GSC availability as an important data-quality constraint. An unavailable GSC value should be interpreted as missing evidence, not as zero search performance.

These checks turn the assumptions in the data contract into measured observations.

In [18]:
# Query 1 — Grain
grain_check = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet('{march_path}')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 10
""").df()

print("🔎 Query 1 — Grain check")
display(grain_check)
print("Duplicate grain combinations:", len(grain_check))


# Query 2 — Row count and date window
march_summary = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date
FROM read_parquet('{march_path}')
""").df()

print("\n📅 Query 2 — Count and date window")
display(march_summary)


# Query 3 — GSC availability
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows
FROM read_parquet('{march_path}')
""").df()

availability_check["availability_rate"] = (
    availability_check["gsc_available_rows"]
    / availability_check["total_rows"]
)

print("\n📊 Query 3 — GSC availability")
display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

🔎 Query 1 — Grain check


,report_date,client_hash_id,content_hash_id,row_count


Duplicate grain combinations: 0

📅 Query 2 — Count and date window


,row_count,first_report_date,last_report_date
0,9841378,2026-03-01,2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


📊 Query 3 — GSC availability


,total_rows,gsc_available_rows,availability_rate
0,9841378,3611061,0.366926


## 4. Data limits

This warehouse is large, but it cannot answer every question about content performance.

**Unbalanced history:** Different clients have different depths of historical data. Therefore, a shorter history should not automatically be interpreted as lower content quality or lower importance.

**Incomplete GSC measurement:** In the March 2026 slice, only 3,611,061 of 9,841,378 observations (36.69%) have `gsc_data_available IS TRUE`. Missing GSC data should be treated as unavailable evidence, not as zero search performance.

**Window overlap:** Some metrics use rolling windows such as 90-day periods. Consecutive observations can therefore share much of the same underlying data and should not be treated as completely independent observations.

**Causality:** The data can identify observed patterns and directional refresh opportunities, but it cannot prove that refreshing a page will cause its future performance to improve. That would require a controlled experiment or stronger causal evidence.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.